#  OBJECTIVE

**This notebook focuses on constructing domain-relevant features from the cleaned flight dataset** to enrich model inputs with meaningful signals — flight duration in minutes, temporal holiday flags, route frequency, mean-price encoding, and polynomial interaction terms — all aimed at improving the predictive accuracy of the flight price model.

> **Input:** `flight_price_cleaned.csv` (10,682 rows, 20 columns) | **Output:** `flight_price_feature_engineered.csv` (35 columns)

Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


base path

In [ ]:
BASE_PATH = "/content/drive/MyDrive/AirFair-Vista"

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv(f"{BASE_PATH}/data/processed/flight_price_cleaned.csv")
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,journey_day,journey_month,journey_weekday,is_weekend,quarter,dep_hour,dep_time_bin,Price_capped,Price_log
0,indigo,2019-03-24,banglore,new delhi,BLR → DEL,22:20:00,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,6,1,1,22,Night,3897,8.268219
1,air india,2019-05-01,kolkata,banglore,CCU → IXR → BBI → BLR,05:50:00,13:15,7h 25m,2 stops,No info,7662,1,5,2,0,2,5,Morning,7662,8.944159
2,jet airways,2019-06-09,delhi,cochin,DEL → LKO → BOM → COK,09:25:00,04:25 10 Jun,19h,2 stops,No info,13882,9,6,6,1,2,9,Morning,13882,9.538420
3,indigo,2019-05-12,kolkata,banglore,CCU → NAG → BLR,18:05:00,23:30,5h 25m,1 stop,No info,6218,12,5,6,1,2,18,Evening,6218,8.735364
4,indigo,2019-03-01,banglore,new delhi,BLR → NAG → DEL,16:50:00,21:35,4h 45m,1 stop,No info,13302,1,3,4,0,1,16,Afternoon,13302,9.495745


---
##  Step: Temporal Feature Extraction & Holiday Flagging

**Why:** The journey date encodes cyclical price signals — weekday vs. weekend pricing and public-holiday surges are well-known demand drivers in aviation. Converting `Date_of_Journey` to numeric weekday and a binary `is_holiday` flag exposes these patterns to the model without needing it to parse raw dates.

TASK 1
Extract temporal features from 'Date_of_Journey'

In [ ]:
df['Date_of_Journey'] = pd.to_datetime(df['Date_of_Journey'])

df['weekday'] = df['Date_of_Journey'].dt.weekday
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0)

In [ ]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,...,journey_day,journey_month,journey_weekday,is_weekend,quarter,dep_hour,dep_time_bin,Price_capped,Price_log,weekday
0,indigo,2019-03-24,banglore,new delhi,BLR → DEL,22:20:00,01:10 22 Mar,2h 50m,non-stop,No info,...,24,3,6,1,1,22,Night,3897,8.268219,6
1,air india,2019-05-01,kolkata,banglore,CCU → IXR → BBI → BLR,05:50:00,13:15,7h 25m,2 stops,No info,...,1,5,2,0,2,5,Morning,7662,8.944159,2
2,jet airways,2019-06-09,delhi,cochin,DEL → LKO → BOM → COK,09:25:00,04:25 10 Jun,19h,2 stops,No info,...,9,6,6,1,2,9,Morning,13882,9.538420,6
3,indigo,2019-05-12,kolkata,banglore,CCU → NAG → BLR,18:05:00,23:30,5h 25m,1 stop,No info,...,12,5,6,1,2,18,Evening,6218,8.735364,6
4,indigo,2019-03-01,banglore,new delhi,BLR → NAG → DEL,16:50:00,21:35,4h 45m,1 stop,No info,...,1,3,4,0,1,16,Afternoon,13302,9.495745,4


In [ ]:
#Holiday Identification
indian_holidays = [
    "2019-01-26",  # Republic Day
    "2019-08-15",  # Independence Day
    "2019-10-02",  # Gandhi Jayanti
]

indian_holidays = pd.to_datetime(indian_holidays)

df['is_holiday'] = df['Date_of_Journey'].isin(indian_holidays).astype(int)

---
##  Step: Duration Parsing → `total_duration_mins` & `flight_type`

**Why:** The raw `Duration` column (e.g., "2h 50m") is a string the model cannot use mathematically. Parsing it into a single numeric column (`total_duration_mins`) gives the model a continuous, ordinal signal strongly correlated with price — longer flights are generally more expensive. Binning into Short/Medium/Long categories adds a human-interpretable categorical signal for tree-based models.

TASK 2
Transform 'Duration' into meaningful metrics


In [ ]:
#Step 1 — Extract hours
df['duration_hours'] = df['Duration'].str.extract('(\d+)h').astype(float)

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-529/255042223.py:2: SyntaxWarning: invalid escape sequence '\d'
  df['duration_hours'] = df['Duration'].str.extract('(\d+)h').astype(float)


In [ ]:
#Step 2 — Extract minutes
df['duration_minutes'] = df['Duration'].str.extract('(\d+)m').astype(float)

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-529/2637483937.py:2: SyntaxWarning: invalid escape sequence '\d'
  df['duration_minutes'] = df['Duration'].str.extract('(\d+)m').astype(float)


In [ ]:
#Step 3 — Fill NaN values
df['duration_hours'] = df['duration_hours'].fillna(0)
df['duration_minutes'] = df['duration_minutes'].fillna(0)

In [ ]:
#Step 4 — Total duration in minutes
df['total_duration_mins'] = df['duration_hours']*60 + df['duration_minutes']

In [ ]:
#Step 5 — Categorize flights
def categorize_flight(x):
    if x < 120:
        return "Short"
    elif x < 300:
        return "Medium"
    else:
        return "Long"

df['flight_type'] = df['total_duration_mins'].apply(categorize_flight)

In [ ]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,...,dep_hour,dep_time_bin,Price_capped,Price_log,weekday,is_holiday,duration_hours,duration_minutes,total_duration_mins,flight_type
0,indigo,2019-03-24,banglore,new delhi,BLR → DEL,22:20:00,01:10 22 Mar,2h 50m,non-stop,No info,...,22,Night,3897,8.268219,6,0,2.0,50.0,170.0,Medium
1,air india,2019-05-01,kolkata,banglore,CCU → IXR → BBI → BLR,05:50:00,13:15,7h 25m,2 stops,No info,...,5,Morning,7662,8.944159,2,0,7.0,25.0,445.0,Long
2,jet airways,2019-06-09,delhi,cochin,DEL → LKO → BOM → COK,09:25:00,04:25 10 Jun,19h,2 stops,No info,...,9,Morning,13882,9.538420,6,0,19.0,0.0,1140.0,Long
3,indigo,2019-05-12,kolkata,banglore,CCU → NAG → BLR,18:05:00,23:30,5h 25m,1 stop,No info,...,18,Evening,6218,8.735364,6,0,5.0,25.0,325.0,Long
4,indigo,2019-03-01,banglore,new delhi,BLR → NAG → DEL,16:50:00,21:35,4h 45m,1 stop,No info,...,16,Afternoon,13302,9.495745,4,0,4.0,45.0,285.0,Medium


---
##  Step: Frequency Encoding — Airline, Source, Destination

**Why:** High-cardinality categorical columns like `Airline` (12 values) cannot simply be one-hot encoded without risking sparsity. Frequency encoding replaces each category with its proportion in the dataset, preserving relative popularity information (e.g., Jet Airways at 36% vs. TruJet at 0.01%). This lets gradient boosting models leverage the size/dominance of each carrier as a price signal.

TASK 3
Frequency Encoding (Airline, Source, Destination)

In [ ]:
#Step 1 — Airline frequency
airline_freq = df['Airline'].value_counts() / len(df)
airline_freq.round(4)

,count
Airline,
jet airways,0.3603
indigo,0.1922
air india,0.1639
multiple carriers,0.1120
spicejet,0.0766
vistara,0.0448
air asia,0.0299
goair,0.0182
multiple carriers premium economy,0.0012


In [ ]:
#Step 2 — Source frequency
source_freq = df['Source'].value_counts() / len(df)
df['Source_freq'] = df['Source'].map(source_freq)
source_freq

,count
Source,
delhi,0.424640
kolkata,0.268770
banglore,0.205673
mumbai,0.065250
chennai,0.035667


In [ ]:
#Step 3 — Destination frequency
dest_freq = df['Destination'].value_counts() / len(df)
df['Destination_freq'] = df['Destination'].map(dest_freq)
dest_freq


,count
Destination,
cochin,0.424640
banglore,0.268770
delhi,0.118424
new delhi,0.087250
hyderabad,0.065250
kolkata,0.035667


---
##  Step: Mean (Target) Encoding — Airline & Source

**Why:** Mean encoding maps each category to the average `Price` associated with it — e.g., Jet Airways Business maps to ~₹11,600 avg price. This directly embeds the price-relationship of each category as a numeric feature, which is one of the most powerful supervised encoding strategies for regression. It must be used carefully to avoid target leakage (handled later via cross-validation splits).

TASK 4
Mean Encoding (Airline & Source based on average ticket price)

In [ ]:
#Airline Mean Encoding
airline_mean = df.groupby('Airline')['Price'].mean()
df['Airline_mean_price'] = df['Airline'].map(airline_mean)

In [ ]:
#Source Mean Encoding
source_mean = df.groupby('Source')['Price'].mean()
df['Source_mean_price'] = df['Source'].map(source_mean)

---
##  Step: Polynomial Feature Generation — `total_duration_mins` × `journey_month`

**Why:** Linear models cannot capture interactions between features. Degree-2 polynomial expansion of `total_duration_mins` and `journey_month` adds squared terms and a cross-product, allowing models to learn non-linear relationships like "long flights in peak months cost disproportionately more." This is especially beneficial before linear baseline models in Notebook 5.

TASK 5
Polynomial Features for Numerical Attributes

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

num_features = ['total_duration_mins','journey_month']

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)

poly_features = poly.fit_transform(df[num_features])

poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(num_features))

df = pd.concat([df, poly_df], axis=1)

In [ ]:
# Set maximum number of columns to 'None' (unlimited)
pd.set_option('display.max_columns', None)
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,journey_day,journey_month,journey_weekday,is_weekend,quarter,dep_hour,dep_time_bin,Price_capped,Price_log,weekday,is_holiday,duration_hours,duration_minutes,total_duration_mins,flight_type,Source_freq,Destination_freq,Airline_mean_price,Source_mean_price,total_duration_mins,journey_month,total_duration_mins^2,total_duration_mins journey_month,journey_month^2
0,indigo,2019-03-24,banglore,new delhi,BLR → DEL,22:20:00,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,6,1,1,22,Night,3897,8.268219,6,0,2.0,50.0,170.0,Medium,0.205673,0.08725,5673.682903,8017.464269,170.0,3.0,28900.0,510.0,9.0
1,air india,2019-05-01,kolkata,banglore,CCU → IXR → BBI → BLR,05:50:00,13:15,7h 25m,2 stops,No info,7662,1,5,2,0,2,5,Morning,7662,8.944159,2,0,7.0,25.0,445.0,Long,0.268770,0.26877,9612.427756,9158.389411,445.0,5.0,198025.0,2225.0,25.0
2,jet airways,2019-06-09,delhi,cochin,DEL → LKO → BOM → COK,09:25:00,04:25 10 Jun,19h,2 stops,No info,13882,9,6,6,1,2,9,Morning,13882,9.538420,6,0,19.0,0.0,1140.0,Long,0.424640,0.42464,11643.923357,10540.113536,1140.0,6.0,1299600.0,6840.0,36.0
3,indigo,2019-05-12,kolkata,banglore,CCU → NAG → BLR,18:05:00,23:30,5h 25m,1 stop,No info,6218,12,5,6,1,2,18,Evening,6218,8.735364,6,0,5.0,25.0,325.0,Long,0.268770,0.26877,5673.682903,9158.389411,325.0,5.0,105625.0,1625.0,25.0
4,indigo,2019-03-01,banglore,new delhi,BLR → NAG → DEL,16:50:00,21:35,4h 45m,1 stop,No info,13302,1,3,4,0,1,16,Afternoon,13302,9.495745,4,0,4.0,45.0,285.0,Medium,0.205673,0.08725,5673.682903,8017.464269,285.0,3.0,81225.0,855.0,9.0


In [ ]:
df.to_csv(f"{BASE_PATH}/data/processed/flight_price_feature_engineered.csv", index=False)

---
##  Next Step → Notebook 03: Exploratory Data Analysis (EDA)

The feature-engineered dataset now has **35 columns** including rich temporal, encoding, and interaction features. **Notebook 03** will visually analyse these features — distributions, correlations, price patterns by airline/stop/duration — to confirm which features carry genuine predictive power before they enter the modelling pipeline.